# Week 7 Assignment: Incremental Data Processing using Delta Lake

### Objectives:
1. Initialize a Spark session configured for Delta Lake.
2. Load and clean the base customer dataset.
3. Save the base dataset as a Delta Table.
4. Read new/incremental updates.
5. Apply a Delta Lake `MERGE` operation (SCD Type 1) to update existing records and insert new ones.
6. Validate results for duplicates and total row counts.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from delta.tables import DeltaTable
import os

# 1. Initialize Spark Session with Delta Lake Configs
spark = SparkSession.builder \
    .appName("DeltaLakeSCDAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark Session initialized with Delta support.")

### Step 1: Load Base Dataset (`customer_master.csv`)
We load the initial customer records from the raw data files.

In [ ]:
# Define local data paths
data_dir = "../data/"
master_path = os.path.join(data_dir, "customer_master.csv")

# Load master dataset
master_df = spark.read.csv(master_path, header=True, inferSchema=True)
print("Base Dataset Loaded:")
master_df.show()

### Step 2: Perform Data Cleaning
We handle null values (e.g. missing emails) and remove duplicate customer records to ensure clean base data.

In [ ]:
# Fill null emails with placeholder and drop duplicates on customer_id
cleaned_master_df = master_df \
    .fillna({"email": "unknown@example.com"}) \
    .dropDuplicates(["customer_id"])

print("Cleaned Base Dataset:")
cleaned_master_df.show()

# Save cleaned base dataset as Delta Table
delta_path = "/tmp/delta-lake/customer_master_table"
cleaned_master_df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Delta table successfully written to: {delta_path}")

### Step 3: Load Incremental Updates (`customer_incremental.csv`)
We load the dataset containing updates to existing customers and new registrations.

In [ ]:
incremental_path = os.path.join(data_dir, "customer_incremental.csv")
incremental_df = spark.read.csv(incremental_path, header=True, inferSchema=True)

print("Incremental Updates Dataset:")
incremental_df.show()

### Step 4: Apply Delta `MERGE` (SCD Type 1)
We merge the incremental data into the base Delta table, updating existing matching IDs and inserting new ones.

In [ ]:
# Load Delta table reference
deltaTable = DeltaTable.forPath(spark, delta_path)

# Execute MERGE operation
deltaTable.alias("target") \
  .merge(
    incremental_df.alias("source"),
    "target.customer_id = source.customer_id"
  ) \
  .whenMatchedUpdate(set = {
    "customer_name": "source.customer_name",
    "email": "source.email",
    "city": "source.city",
    "last_updated": "source.last_updated"
  }) \
  .whenNotMatchedInsert(values = {
    "customer_id": "source.customer_id",
    "customer_name": "source.customer_name",
    "email": "source.email",
    "city": "source.city",
    "last_updated": "source.last_updated"
  }) \
  .execute()

print("Delta MERGE operation completed.")

### Step 5: Validate Results
Validate that there are no duplicate `customer_id` keys in the merged Delta table and verify the expected row count.

In [ ]:
# Read final merged data
final_df = spark.read.format("delta").load(delta_path)

# 1. Row count validation
total_rows = final_df.count()
print(f"Total Row Count in Merged Table: {total_rows}")

# 2. Duplicate validation
unique_ids = final_df.select("customer_id").distinct().count()
assert total_rows == unique_ids, "Duplicate customer_id entries detected!"
print("Validation successful: No duplicates found.")

### Step 6: Display Final Table State
Display the resulting dataset demonstrating updated records and new insertions.

In [ ]:
print("Final Merged Delta Table:")
final_df.orderBy("customer_id").show()